# Driver Drowsiness Detection — Week 4
### Model Training Phase 1 — Baseline Run
NTCC | Amity School of Engineering & Technology | May 2026

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, BatchNormalization,
    Dropout, Flatten, Dense
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (
    ModelCheckpoint, EarlyStopping,
    ReduceLROnPlateau, CSVLogger
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

DRIVE  = '/content/drive/MyDrive/NTCC_Drowsiness_Project'
RES    = f'{DRIVE}/results'
MODELS = f'{DRIVE}/models'
os.makedirs(RES,    exist_ok=True)
os.makedirs(MODELS, exist_ok=True)

IMG_SIZE    = 64
NUM_CLASSES = 4
BATCH_SIZE  = 32

print('TF:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
GITHUB_USERNAME = 'Nimit15'
GITHUB_TOKEN    = 'paste_your_token_here'
GITHUB_EMAIL    = 'your_email@gmail.com'
REPO_NAME       = 'Driver-Drowsiness-Detection-Using-Deep-Learning-Techniques'
REPO_PATH       = f'/content/{REPO_NAME}'

os.system(f'git config --global user.email "{GITHUB_EMAIL}"')
os.system(f'git config --global user.name "{GITHUB_USERNAME}"')

if not os.path.exists(REPO_PATH):
    url = f'https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git'
    os.system(f'git clone {url} {REPO_PATH}')
else:
    os.chdir(REPO_PATH)
    os.system('git pull')
print('Repo ready')

## Load preprocessed data

In [ ]:
X_train = np.load(f'{DRIVE}/data/X_train.npy')
X_val   = np.load(f'{DRIVE}/data/X_val.npy')
X_test  = np.load(f'{DRIVE}/data/X_test.npy')
y_train = np.load(f'{DRIVE}/data/y_train.npy')
y_val   = np.load(f'{DRIVE}/data/y_val.npy')
y_test  = np.load(f'{DRIVE}/data/y_test.npy')

CLASS_NAMES = ['Closed', 'Open', 'no_yawn', 'yawn']

print('Train:', X_train.shape, '| Val:', X_val.shape, '| Test:', X_test.shape)

## Rebuild CNN from Week 3

In [ ]:
model = Sequential([
    Conv2D(32, (3,3), activation='relu', padding='same',
           input_shape=(IMG_SIZE, IMG_SIZE, 1)),
    BatchNormalization(),
    Conv2D(32, (3,3), activation='relu', padding='same'),
    MaxPooling2D(2,2),
    Dropout(0.25),

    Conv2D(64, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    Conv2D(64, (3,3), activation='relu', padding='same'),
    MaxPooling2D(2,2),
    Dropout(0.25),

    Conv2D(128, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(2,2),
    Dropout(0.25),

    Flatten(),
    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.50),
    Dense(NUM_CLASSES, activation='softmax')
], name='Drowsiness_CNN')

model.compile(
    optimizer = Adam(learning_rate=0.0005),
    loss      = 'categorical_crossentropy',
    metrics   = ['accuracy']
)
model.summary()

## Augmentation + callbacks

In [ ]:
train_gen = ImageDataGenerator(
    rotation_range=10, horizontal_flip=True,
    brightness_range=[0.8,1.2], zoom_range=0.1,
    width_shift_range=0.1, height_shift_range=0.1
).flow(X_train, y_train, batch_size=BATCH_SIZE, shuffle=True)

val_gen = ImageDataGenerator().flow(
    X_val, y_val, batch_size=BATCH_SIZE, shuffle=False)

cbs = [
    ModelCheckpoint(f'{MODELS}/best_model.h5',
                    monitor='val_accuracy', save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=15,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                      patience=5, min_lr=1e-7, verbose=1),
    CSVLogger(f'{RES}/training_log.csv')
]

print('Steps/epoch :', len(X_train) // BATCH_SIZE)
print('Val steps   :', len(X_val)   // BATCH_SIZE)

## Training — baseline run (30 epochs)

In [ ]:
history = model.fit(
    train_gen,
    steps_per_epoch  = len(X_train) // BATCH_SIZE,
    epochs           = 30,
    validation_data  = val_gen,
    validation_steps = len(X_val) // BATCH_SIZE,
    callbacks        = cbs,
    verbose          = 1
)

print('Done')
print(f'Best val_accuracy: {max(history.history["val_accuracy"]):.4f}')

## Evaluation on test set

In [ ]:
best = load_model(f'{MODELS}/best_model.h5')
test_loss, test_acc = best.evaluate(X_test, y_test, verbose=0)
print(f'Test Accuracy : {test_acc:.4f} ({test_acc*100:.2f}%)')
print(f'Test Loss     : {test_loss:.4f}')

In [ ]:
log = pd.read_csv(f'{RES}/training_log.csv')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Baseline Training — Week 4', fontsize=12)

axes[0].plot(log['epoch']+1, log['accuracy'],
             label='Train', color='#1E88E5', lw=2)
axes[0].plot(log['epoch']+1, log['val_accuracy'],
             label='Val', color='#E53935', lw=2, linestyle='--')
axes[0].set_title('Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].set_ylim(0, 1.05)
axes[0].grid(alpha=0.3)

axes[1].plot(log['epoch']+1, log['loss'],
             label='Train', color='#1E88E5', lw=2)
axes[1].plot(log['epoch']+1, log['val_loss'],
             label='Val', color='#E53935', lw=2, linestyle='--')
axes[1].set_title('Loss')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RES}/training_curves_week4.png', dpi=150)
plt.show()

In [ ]:
y_pred = np.argmax(best.predict(X_test, verbose=0), axis=1)
y_true = np.argmax(y_test, axis=1)

cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(
    ax=ax, cmap='Blues', colorbar=True)
ax.set_title('Confusion Matrix — Test Set', fontsize=12, pad=12)
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.savefig(f'{RES}/confusion_matrix_week4.png', dpi=150)
plt.show()

print(classification_report(y_true, y_pred,
                             target_names=CLASS_NAMES, digits=4))

In [ ]:
log = pd.read_csv(f'{RES}/training_log.csv')

print('='*50)
print('WEEK 4 BASELINE TRAINING RESULTS')
print('='*50)
print(f'Epochs trained    : {len(log)}')
print(f'Best val_accuracy : {max(log["val_accuracy"]):.4f} ({max(log["val_accuracy"])*100:.2f}%)')
print(f'Best val_loss     : {min(log["val_loss"]):.4f}')
print(f'Final train_acc   : {log["accuracy"].iloc[-1]:.4f}')
print(f'Test accuracy     : {test_acc:.4f} ({test_acc*100:.2f}%)')
print(f'Test loss         : {test_loss:.4f}')
print()
print('Note: Low accuracy observed — will investigate in Week 5')
print('Next: Check data loading, labels and try MobileNetV2')
print('='*50)

In [ ]:
import shutil
from datetime import datetime

NOTEBOOK = 'Week4_Training_Baseline.ipynb'
os.makedirs(f'{REPO_PATH}/notebooks', exist_ok=True)
os.makedirs(f'{REPO_PATH}/results',   exist_ok=True)

shutil.copy(f'/content/{NOTEBOOK}', f'{REPO_PATH}/notebooks/{NOTEBOOK}')

for fname in ['training_curves_week4.png', 'confusion_matrix_week4.png', 'training_log.csv']:
    src = f'{RES}/{fname}'
    if os.path.exists(src):
        shutil.copy(src, f'{REPO_PATH}/results/{fname}')

os.chdir(REPO_PATH)
os.system('git add .')
os.system(f'git commit -m "Week 4: Baseline training run — {datetime.now().strftime("%d %b %Y")}'+'"')
print(os.popen('git push 2>&1').read())